# Part 3: Agent-driven policy research

Build a research system in `research/` in which LLM agents propose serving-policy changes, implement them, evaluate them in the simulator, and decide what to try next; then select a final policy from an unattended campaign. Two questions matter as much as the speedup: what the agents add beyond the policies you can build yourself, and what the selected policy actually does.

![One possible research system and the evaluation splits.](../docs/figures/research_system.svg)

Keep your system and its outputs in files so campaigns run from the command line:

```text
research/      your research system, evaluator, prompts, and configuration
candidates/    every seed and generated candidate, including failures
runs/          campaign logs with lineage, scores, token usage, and interventions
```

Use this notebook to inspect workloads, score seeds, test your evaluator, and analyze campaigns. The [Part 3 handout](../docs/part3_handout.md) has the rules and the report questions.

| Section | Task | Estimated time |
|---|---|---|
| 1–3 | Metric, workloads, seeds, and evaluator | 2 h |
| 4 | Build your research system; run and ablate campaigns | 8–10 h, much of it unattended |
| 5–6 | Select finalists; explain and ablate the selected policy | 2 h |
| Questions | Answer Q1–Q6 | 3–4 h |

**Grading (60%):** policy performance on the hidden test set 30%; research report Q1–Q6 30% (10% for each pair of questions). Extra credit: up to +10% for a top-3% hidden-test speedup or an exceptional research system or report.

## Before you begin

Complete the [environment setup](../README.md#setup) and select the **CS 378 Assignment 4** kernel. Part 3 runs every policy on two Llama-3-8B replicas, each a simulated A100 with its own queue and KV cache, starting from empty caches.

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt

from agentkv import viz
from agentkv.notebook import check, repo_root

os.chdir(repo_root())          # every path below is relative to the repository root
plt.rcParams["figure.dpi"] = 100

from agentkv import metric, score
from agentkv.trace import load

DEV = sorted(Path("data/traces/public_dev").glob("*.json"))
print(f"{len(DEV)} dev traces; full credit at S = {metric.FULL_CREDIT_SPEEDUP:.3f}")

## 1. Performance metric

Speedup on a trace is the reference's mean job latency divided by yours. `S` takes a geometric mean within each family and then across the eight families, so every family counts equally.

```text
g      = 1 - 1/S
g_eff  = g - sum over families of max(0, 0.90 - S_f)
points = clip(10 + 20 × g_eff / 0.09, 0, 30)
```

Performance is worth 30% of the grade. A valid policy that matches the reference earns 10%; `S ≥ 1.099` with every family at or above 0.90 earns the full 30%. A family below 0.90 costs credit in proportion to its shortfall. Your policy cannot see family labels or trace names; only your research system can.

In [ ]:
reference = score("policies/reference.py", split="public_dev")
print(reference.summary())
viz.families(reference)
plt.show()

## 2. The eight workload families

![Workload families: call structure and which parts of a prompt later calls can reuse.](../docs/figures/families.svg)

The table compares calls per job, prompt and output lengths, the share of prompt tokens in previously seen blocks, and each trace's KV capacity per replica.

In [ ]:
def summarize(path):
    t = load(path)
    calls = [n for _, n in t.llm_nodes()]
    seen, shared, total = set(), 0, 0
    for job in sorted(t.jobs, key=lambda j: j.arrival):
        for n in (x for x in job.nodes if x.type == "llm"):
            full = n.hashes[: n.n_prompt // t.memory.block_size]
            shared += sum(h in seen for h in full) * t.memory.block_size
            total += n.n_prompt
            seen.update(full)
    return {"trace": t.trace_id, "calls/job": len(calls) / len(t.jobs),
            "prompt": sum(n.n_prompt for n in calls) / len(calls),
            "output": sum(n.n_output for n in calls) / len(calls),
            "shared": shared / total, "kv tokens": t.memory.num_blocks * t.memory.block_size,
            "description": t.description}

rows = [summarize(p) for p in DEV]
print(f"{'trace':10} {'calls/job':>9} {'prompt':>8} {'output':>8} {'shared':>7} {'kv tokens':>10}")
for r in rows:
    print(f"{r['trace']:10} {r['calls/job']:9.1f} {r['prompt']:8.0f} {r['output']:8.0f} "
          f"{r['shared']:7.0%} {r['kv tokens']:10,d}")
print()
for r in rows[::4]:
    print(f"{r['trace']}: {r['description']}")

`viz.record(..., replicas=2)` replays one trace and shows where time goes: waiting for a slot, waiting for admission, or running, with memory below. You will use it for Q5 and Q6.

In [ ]:
rec = viz.record("data/traces/public_dev/dev-W4-0.json", "policies/prefix_policy.py", replicas=2)
fig, axes = plt.subplots(2, 1, figsize=(12, 10), gridspec_kw={"height_ratios": [2.4, 1]},
                         layout="constrained")
viz.timeline(rec, axes[0])
viz.memory(rec, axes[1])
plt.show()

## 3. Seeds and the evaluator

A campaign may start from any policies you choose. Score your candidate seeds first: they are the baseline that Q2 measures your agents against. `split="screen"` (one trace per family) is a quarter of the cost of `public_dev`.

In [ ]:
seeds = {"reference": ("policies/reference.py", None), "FIFO": ("policies/fifo.py", None),
         "prefix policy": ("policies/prefix_policy.py", None)}
for name, (path, params) in seeds.items():
    quick = score(path, split="screen", params=params)
    dev = score(path, split="public_dev", params=params)
    print(f"{name:14} screen S = {quick.speedup:.3f}   public_dev S = {dev.speedup:.3f}   "
          f"points {dev.points:.1f}")

Copy `examples/evaluator.py` to `research/evaluator.py` and choose the objective your system maximizes and the feedback your agents see. The example returns the score dict, a `diagnostics` text with per-trace cache and preemption statistics, and an `objective` that is `None` for invalid candidates. Keep `speedup`, `by_family`, and `valid` unchanged.

In [ ]:
import runpy

evaluator = runpy.run_path("examples/evaluator.py")      # research/evaluator.py once you have adapted it
feedback = evaluator["evaluate"]("policies/prefix_policy.py", split="screen")
print("objective:", feedback["objective"])
print(feedback["diagnostics"][:1500])

## 4. Build and run your research system

`examples/research_system.py` is a minimal greedy loop: one prompt with the API, the evaluator, the current best policy, and a one-line history; one code block back; keep the best. Build your own in `research/`, or adapt OpenEvolve, ShinkaEvolve, or CORAL. Questions worth designing around:

- **Hypotheses and diagnostics.** Do agents propose better changes when they see per-family speedups and per-trace statistics, or when planning is separated from coding?
- **Failures.** Candidates fail on syntax, sandbox rules, the 50 ms limit, or the 4 MiB state limit. Does your system repair them using the violation message?
- **Exploration.** A greedy single lineage stalls on sharp optima: in Part 2, routing by cache affinity helps only with a bounded slack. Do multiple lineages, parameter sweeps, or an archive of diverse policies help?
- **Evaluation budget.** Screening before scoring `public_dev` saves most of the simulation time.

Run the example offline to see the log format (`log.jsonl`, `calls.jsonl`, `campaign.json`, `selection.json`) that the plots below read. Write the same records from your own system.

In [ ]:
from datetime import datetime

USE_BEDROCK = False
EVALUATOR = Path("examples/evaluator.py")             # research/evaluator.py once you have adapted it
DEMO_RUN = Path("runs") / f"example-demo-{datetime.now():%Y%m%d-%H%M%S-%f}"
OFFLINE = "" if USE_BEDROCK else "--offline"
!{sys.executable} examples/research_system.py --evaluator {EVALUATOR} --iters 3 {OFFLINE} --out {DEMO_RUN}

`campaign_plots` shows the best development speedup so far against candidates scored and tokens spent, starting from the campaign's seed. Pass validation scores of finalists to mark them. Use it for Q4.

In [ ]:
from matplotlib.ticker import MaxNLocator


def campaign_plots(log_path, start_S=1.0, finalists=None):
    rows = [json.loads(line) for line in Path(log_path).read_text().splitlines() if line.strip()]
    rows = [r for r in rows if not r.get("finalist")]
    best, tokens = start_S, 0
    xs, ts, bests, devs = [0], [0], [start_S], []
    for i, r in enumerate(rows, 1):
        tokens += r.get("tokens", 0)
        if r.get("valid") and r.get("dev_S"):
            best = max(best, r["dev_S"])
            devs.append((i, tokens, r["dev_S"]))
        xs.append(i); ts.append(tokens); bests.append(best)
    valid = sum(bool(r.get("valid")) for r in rows) / max(len(rows), 1)
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), sharey=True, layout="constrained")
    for ax, x, key in ((axes[0], xs, 0), (axes[1], ts, 1)):
        ax.step(x, bests, where="post", color=viz.COLORS["running"], label="best so far")
        ax.scatter([d[key] for d in devs], [d[2] for d in devs], s=12, color="#999999", label="candidate")
        for name, (cand, val_S) in (finalists or {}).items():
            point = next((d for d in devs if rows[d[0] - 1]["id"] == cand), None)
            if point:
                ax.scatter(point[key], val_S, marker="*", s=90, color=viz.COLORS["critical"], zorder=3)
                ax.annotate(f"{name} (val)", (point[key], val_S), fontsize=8, xytext=(3, 3),
                            textcoords="offset points")
        ax.axhline(1.0, color="gray", lw=0.8)
    axes[0].xaxis.set_major_locator(MaxNLocator(integer=True))
    if tokens == 0:
        axes[1].text(0.5, 0.5, "no LLM tokens (offline run)", transform=axes[1].transAxes, ha="center")
    axes[0].set(xlabel="candidates scored", ylabel="public_dev S")
    axes[1].set(xlabel="LLM tokens")
    axes[0].legend(frameon=False, fontsize=8)
    fig.suptitle(f"{log_path}: {len(rows)} candidates, {valid:.0%} valid")
    plt.show()


campaign_plots(DEMO_RUN / "log.jsonl", start_S=reference.speedup)    # then your own campaigns

## 5. Finalists

Search on `public_dev`; score only finalists on `public_val`. A policy valid on one split can still fail on another, and with two to four traces per family the same policy scores a few percent differently across splits, so compare both and report the spread.

In [ ]:
finalists = {
    # "c17": "candidates/<campaign>/c17.py",
}
results = {name: (score(path, split="public_dev"), score(path, split="public_val"))
           for name, path in finalists.items()}
if results:
    fig, ax = plt.subplots(figsize=(5, 4.5))
    for name, (dev, val) in results.items():
        print(f"{name}: dev S = {dev.speedup:.4f}, val S = {val.speedup:.4f}, "
              f"valid on both = {dev.valid and val.valid}")
        ax.scatter(dev.speedup, val.speedup)
        ax.annotate(name, (dev.speedup, val.speedup), fontsize=8, xytext=(3, 3), textcoords="offset points")
    ax.axline((1, 1), slope=1, color="gray", ls="--", lw=0.8)
    ax.set(xlabel="public_dev S", ylabel="public_val S")
    plt.show()

## 6. Understand and ablate the selected policy

Read the selected policy line by line and write its logic as pseudocode. Agents often leave dead code and give plausible explanations that the measurements do not support, so disable one mechanism at a time in a copy (or through a parameter) and measure the change on both public splits. Use `viz.record` on one trace to show a decision the policy makes differently from the reference.

In [ ]:
selected = "candidates/<campaign>/<id>.py"
ablations = {
    # "no affinity routing": ("candidates/<campaign>/<id>_no_routing.py", None),
}
if Path(selected).exists():
    full = {split: score(selected, split=split) for split in ("public_dev", "public_val")}
    for name, (path, params) in ablations.items():
        for split in full:
            r = score(path, split=split, params=params)
            change = " ".join(f"{f} {r.by_family[f] / full[split].by_family[f]:.3f}" for f in r.by_family)
            print(f"{name:24} {split:10} S = {r.speedup:.4f} (full {full[split].speedup:.4f})  {change}")

## Questions

Answer Q1–Q6 in `assignment4_report.pdf` after Parts 1 and 2, each below its question; the full questions are in the [Part 3 handout](../docs/part3_handout.md#research-report). Cite run ids and candidate hashes.

1. **How does your research system work?**
2. **What do the agents contribute?** Best seed alone, a campaign from the reference, a campaign from your seed, and one ablation, all with the same model and token budget.
3. **How autonomous was it?**
4. **Did more spending help?**
5. **What policy did the agents find?**
6. **Where does it fail?**

## Submission checklist

- Gradescope and Canvas: `policy.py`, optional `params.json`, and `submission.json` with the policy's SHA-256 (`r.policy_sha256`), Python version, and non-standard-library imports.
- Canvas: `research/`, `candidates/` (seeds and every candidate, including failures), and `runs/` (campaign logs, token receipts, and intervention logs).
- Canvas: `assignment4_report.pdf` with all parts' answers in order, run ids, candidate hashes, the `pytest` summary, and token totals.